# PatchTST Pretraining Optimization - Google Colab

This notebook performs hyperparameter optimization for self-supervised pretraining of the PatchTST encoder using masked patch reconstruction.

**Prerequisites**: Modify `src/config/tuning_pretrain_config.py` before execution.

**Output**: Pretrained encoder weights saved to Google Drive.

## 1. Environment Setup

Mount Google Drive and clone the repository.

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive

drive.mount("/content/drive")

# Define results directory on Drive
import os

DRIVE_RESULTS_DIR = "/content/drive/MyDrive/pretrain_results"
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
print(f"Results directory: {DRIVE_RESULTS_DIR}")

In [ ]:
# Clone or update repository
REPO_PATH = "/content/Progetto_deep_learning"
BRANCH = "develop"

if not os.path.exists(REPO_PATH):
    print(f"Cloning branch {BRANCH}...")
    !git clone -b {BRANCH} https://github.com/scorzaluca/Progetto_deep_learning.git
else:
    print(f"Repository exists, updating branch {BRANCH}...")
    !cd {REPO_PATH} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}

# Install dependencies and configure Python path
%pip install optuna transformers -q
import sys

sys.path.insert(0, REPO_PATH)

print("\nSetup complete.")
print("\nIMPORTANT: Modify configuration file before proceeding:")
print(f"   {REPO_PATH}/src/config/tuning_pretrain_config.py")

## Configuration Instructions

Before running the optimization, update the configuration file:

1. Click the **folder icon** in the left sidebar
2. Navigate to: `Progetto_deep_learning/src/config/`
3. Double-click `tuning_pretrain_config.py` to open
4. Modify parameters and **save** (Ctrl+S)
5. **Re-run the cell below** to reload modules

In [ ]:
# Re-run this cell after any configuration file modifications

import importlib
import src.config.tuning_pretrain_config as tpc
import src.config.model_config as mc
import src.config.training_config as trc
import src.config as cfg

# Reload all config modules
importlib.reload(tpc)
importlib.reload(mc)
importlib.reload(trc)
importlib.reload(cfg)

# Import updated configuration
from src.config.tuning_pretrain_config import (
    N_TRIALS,
    EPOCHS,
    PATIENCE,
    STUDY_NAME,
    NEW_STUDY,
    VAL_SPLIT,
    DATA_PATH,
)
from src.config import SEED, LOOKBACK, INPUT_SIZE

# Verify GPU availability
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Study: {STUDY_NAME} ({'NEW' if NEW_STUDY else 'RESUME'})")
print(f"Trials: {N_TRIALS}, Epochs: {EPOCHS}, Patience: {PATIENCE}")
print(f"Validation split: {VAL_SPLIT}")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("\nConfiguration reloaded successfully.")

## 2. Load Data

Load and normalize the dataset, then create train/validation DataLoaders for pretraining.

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader
from src.DataLoading import PretrainingDataset
from src.Utils import set_seed

set_seed(SEED)

# Load and prepare data
data_path = f"{REPO_PATH}/{DATA_PATH}"
df = pd.read_csv(data_path)

split_idx = int(len(df) * (1 - VAL_SPLIT))
df_train = df.iloc[:split_idx]
df_val = df.iloc[split_idx:]

# Normalize data
scaler = MinMaxScaler()
df_train_scaled = pd.DataFrame(scaler.fit_transform(df_train), columns=df.columns)
df_val_scaled = pd.DataFrame(scaler.transform(df_val), columns=df.columns)

# Create DataLoaders
train_dataset = PretrainingDataset(df=df_train_scaled, lookback=LOOKBACK, step=1)
val_dataset = PretrainingDataset(df=df_val_scaled, lookback=LOOKBACK, step=1)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"Train samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")

## 3. Optuna Optimization

Define the hyperparameter search space and run optimization.

In [ ]:
import optuna
from optuna.samplers import TPESampler
from torch.optim.lr_scheduler import ReduceLROnPlateau
from src.ModelClasses import PatchTSTPretraining


def get_pretrain_space(trial):
    """Define the hyperparameter search space."""
    return {
        "d_model": trial.suggest_categorical("d_model", [64, 128, 256]),
        "n_heads": trial.suggest_categorical("n_heads", [2, 4, 8]),
        "n_layers": trial.suggest_int("n_layers", 2, 4),
        "patch_length": trial.suggest_categorical("patch_length", [8, 12, 16, 24]),
        "stride": trial.suggest_categorical("stride", [4, 6, 8]),
        "mask_ratio": trial.suggest_float("mask_ratio", 0.3, 0.6),
        "dropout": trial.suggest_float("dropout", 0.1, 0.3),
        "lr": trial.suggest_float("lr", 1e-5, 1e-3, log=True),
    }


def fit_pretrain(model, train_loader, val_loader, epochs, lr, device, patience):
    """Train the pretraining model with early stopping."""
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-6
    )

    best_val_loss = float("inf")
    epochs_no_improve = 0
    best_encoder_state = None

    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        for batch_x in train_loader:
            batch_x = batch_x.to(device)
            optimizer.zero_grad()
            loss = model(batch_x)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_x in val_loader:
                batch_x = batch_x.to(device)
                loss = model(batch_x)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        scheduler.step(val_loss)

        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            best_encoder_state = model.get_encoder_state_dict()
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    return best_encoder_state, best_val_loss


# Global variables to track best encoder
best_encoder_state_global = None
best_loss_global = float("inf")


def objective(trial):
    """Optuna objective function for pretraining optimization."""
    global best_encoder_state_global, best_loss_global

    params = get_pretrain_space(trial)
    model_config = {"num_channels": INPUT_SIZE, "lookback": LOOKBACK, **params}

    model = PatchTSTPretraining(model_config)
    encoder_state, val_loss = fit_pretrain(
        model, train_loader, val_loader, EPOCHS, params["lr"], DEVICE, PATIENCE
    )

    if val_loss < best_loss_global:
        best_loss_global = val_loss
        best_encoder_state_global = encoder_state

    return val_loss


print("Optimization objective defined. Ready to run.")

In [ ]:
# Run optimization
STORAGE_PATH = f"{DRIVE_RESULTS_DIR}/pretrain_studies.db"

if NEW_STUDY:
    try:
        optuna.delete_study(study_name=STUDY_NAME, storage=f"sqlite:///{STORAGE_PATH}")
        print(f"Existing study '{STUDY_NAME}' deleted.")
    except:
        pass

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=f"sqlite:///{STORAGE_PATH}",
    load_if_exists=not NEW_STUDY,
    direction="minimize",
    sampler=TPESampler(seed=SEED),
)

remaining = max(0, N_TRIALS - len(study.trials))
print(f"Completed trials: {len(study.trials)}, Remaining: {remaining}")

if remaining > 0:
    study.optimize(objective, n_trials=remaining, show_progress_bar=True)

print(f"\nBest validation loss: {study.best_value:.6f}")
print(f"Best parameters: {study.best_params}")

## 4. Save Results

Save the pretrained encoder weights and best hyperparameters to Google Drive.

In [ ]:
import json

# Save encoder weights
encoder_path = f"{DRIVE_RESULTS_DIR}/{STUDY_NAME}_encoder.pth"
torch.save(best_encoder_state_global, encoder_path)
print(f"Encoder saved: {encoder_path}")

# Save hyperparameters
params_path = f"{DRIVE_RESULTS_DIR}/{STUDY_NAME}_params.json"
results = {
    "study_name": STUDY_NAME,
    "best_val_loss": study.best_value,
    "best_params": study.best_params,
}
with open(params_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Parameters saved: {params_path}")

print("\n" + "=" * 50)
print("NEXT STEPS:")
print("=" * 50)
print(f"1. Update d_model={study.best_params.get('d_model', 128)} in PATCHTST_CONFIG")
print(f"2. Copy the encoder to Google Drive for future use")
print(f"3. For EncoderLSTM, use: pretrain_path='{encoder_path}'")

## 5. Visualization

Plot the optimization history showing trial progression and convergence.

In [ ]:
import matplotlib.pyplot as plt

values = [t.value for t in study.trials if t.value is not None]
best_values = [min(values[: i + 1]) for i in range(len(values))]

plt.figure(figsize=(10, 4))
plt.plot(values, "o-", alpha=0.6, label="Trial Loss")
plt.plot(best_values, "r-", linewidth=2, label="Best Loss")
plt.xlabel("Trial")
plt.ylabel("Reconstruction Loss")
plt.title(f"{STUDY_NAME} - Optimization History")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(f"{DRIVE_RESULTS_DIR}/{STUDY_NAME}_plot.png", dpi=150)
plt.show()

print(f"\nPlot saved: {DRIVE_RESULTS_DIR}/{STUDY_NAME}_plot.png")